Extracting the AutoCAD data and loading it into the duckdb database.
=

In [1]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\reask"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\reask\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\reask\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Reask - Eletromecânico 2.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Reask - Eletromecânico 2.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Reask - Eletromecânico 2.dwg")

conn = load_to_duckdb(dxf_file, "PV Reask")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Reask already exists
Removed existing data for project: PV Reask
Successfully loaded project
Loaded 19289 entities for project: PV Reask
Don't forget to close connection.
Data Loaded
DuckDB Loaded


Loading the data from PV Reask
=

In [2]:
data_ELM = conn.sql("SELECT layer, name, X, Y, UFV, PST, INV, GRP, STR, rotation FROM cad_entities WHERE project_name = 'PV Reask'")

print(data_ELM)

result = conn.execute("""
    SELECT
        layer,
        COUNT(*) as count
    FROM cad_entities
    GROUP BY layer
    ORDER BY count
""")

print(result)

┌─────────────────────────┬───────────────────────┬───────────┬───────────┬─────────┬─────────┬─────────┬─────────┬─────────┬──────────┐
│          layer          │         name          │     X     │     Y     │   UFV   │   PST   │   INV   │   GRP   │   STR   │ rotation │
│         varchar         │        varchar        │   float   │   float   │ varchar │ varchar │ varchar │ varchar │ varchar │  float   │
├─────────────────────────┼───────────────────────┼───────────┼───────────┼─────────┼─────────┼─────────┼─────────┼─────────┼──────────┤
│ EMF_em_pst_1.1          │ EMF422-1_eletrocentro │ 700595.75 │  756504.5 │         │         │ NULL    │ NULL    │ NULL    │      0.0 │
│ EMF_em_pst_1.3          │ EMF422-1_eletrocentro │  700624.3 │ 756297.44 │         │         │ NULL    │ NULL    │ NULL    │    180.0 │
│ EMF_em_pst_1.2          │ EMF422-1_eletrocentro │  700364.6 │ 756310.75 │         │         │ NULL    │ NULL    │ NULL    │      0.0 │
│ EMF_em_pst_1.4          │ EMF422-1_elet

Separating the data into different dataframes.
=
First one is for strings.

In [3]:
strings = conn.sql("SELECT * FROM data_ELM WHERE name LIKE 'EMF422-1_string_%'")

strings_df = strings.fetchdf()
print(strings)
print(strings_df.describe())
print(strings_df.info())

┌────────────────┬───────────────────┬───────────┬───────────┬─────────┬─────────┬─────────┬──────────┬─────────┬──────────┐
│     layer      │       name        │     X     │     Y     │   UFV   │   PST   │   INV   │   GRP    │   STR   │ rotation │
│    varchar     │      varchar      │   float   │   float   │ varchar │ varchar │ varchar │ varchar  │ varchar │  float   │
├────────────────┼───────────────────┼───────────┼───────────┼─────────┼─────────┼─────────┼──────────┼─────────┼──────────┤
│ EMF_em_pst_1.1 │ EMF422-1_string_2 │ 700354.75 │ 756686.25 │ 01      │ 01      │ 01      │ 01.01.01 │ 01      │      0.0 │
│ EMF_em_pst_1.1 │ EMF422-1_string_2 │ 700354.75 │ 756686.25 │ 01      │ 01      │ 01      │ 01.01.01 │ 02      │    180.0 │
│ EMF_em_pst_1.1 │ EMF422-1_string_3 │ 700361.25 │  756677.6 │ 01      │ 01      │ 01      │ 01.01.01 │ 03      │      0.0 │
│ EMF_em_pst_1.1 │ EMF422-1_string_3 │ 700361.25 │ 756675.25 │ 01      │ 01      │ 01      │ 01.01.01 │ 04      │      0.0 │


Second one is for the inverters.


In [4]:
inverter = conn.sql("SELECT layer, name, X, Y, UFV, PST, INV, GRP FROM data_ELM WHERE name = 'EMF422-1_inversor'")
print(inverter)

inverter_df = inverter.fetchdf()
inverter_df["UFV"] = inverter_df["UFV"].astype(str).str.zfill(2)
inverter_df['GRP'] = inverter_df["UFV"].astype(str) + "." + inverter_df["PST"].astype(str) + "." + inverter_df["INV"].astype(str)
inverter_df.rename(columns={'X': 'inverter_x', 'Y': 'inverter_y'}, inplace=True)

print(inverter_df.describe())
print(inverter_df.info())
print(inverter_df)

┌────────────────┬───────────────────┬───────────┬───────────┬─────────┬─────────┬─────────┬─────────┐
│     layer      │       name        │     X     │     Y     │   UFV   │   PST   │   INV   │   GRP   │
│    varchar     │      varchar      │   float   │   float   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼───────────────────┼───────────┼───────────┼─────────┼─────────┼─────────┼─────────┤
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700436.25 │ 756667.25 │ 01      │ 01      │ 01      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700520.25 │ 756634.25 │ 01      │ 01      │ 02      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700595.25 │ 756601.25 │ 01      │ 01      │ 04      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700597.94 │ 756568.25 │ 01      │ 01      │ 06      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700597.94 │ 756535.25 │ 01      │ 01      │ 09      │         │
│ EMF_em_pst_1.1 │ EMF422-1_inversor │ 700597.94 │ 756524.25 │ 01      │ 

Comparing the coordinates of the inverter to the mean of the X coordinate from all the strings that share a GRP with the inverter, in order to determine whether is the inverter is to the west (W) or east (E) in relation to their strings.
=

In [5]:
struct_df = strings_df.copy()
struct_df['Comparing X'] = struct_df.groupby('GRP')['X'].transform('mean')
print(f'1:{len(struct_df)}')
struct_df = struct_df.merge(inverter_df, on='GRP')
print(f'2:{len(struct_df)}')
struct_df['INVERTER - E or W'] = np.where(
    struct_df['inverter_x'] >= struct_df['Comparing X'],
    'E',
    'W'
)
inverter_position = struct_df[['GRP', 'INVERTER - E or W']].drop_duplicates("GRP").sort_values(by='GRP').reset_index(drop=True)
print(inverter_position)

1:5488
2:5461
          GRP INVERTER - E or W
0    01.01.01                 E
1    01.01.02                 E
2    01.01.03                 E
3    01.01.04                 E
4    01.01.05                 E
..        ...               ...
194  03.03.16                 E
195  03.03.17                 E
196  03.03.18                 E
197  03.03.19                 E
198  03.03.20                 E

[199 rows x 2 columns]


Iterating through the strings dataframe to determine the classification of each string.
=

S = 1-string structure.

U_R = Middle part of structure on the right on a 2-string structure.
U_L = Middle part of structure on the left on a 2-string structure.

T = Top string of a 3-string structure.
M = Middle string of a 3-string structure.
B = Bottom string of a 3-string structure.

In [6]:
for idx, row in strings_df.iterrows():
    x = row['X']
    y = row['Y']


    mask_TMB = (
        (strings_df['Y'] >= y - 5) &
        (strings_df['Y'] <= y + 5) &
        (strings_df['X'] >= x - 0.5) &
        (strings_df['X'] <= x + 0.5) &
        (strings_df['Y'] != y) &
        (strings_df.index != idx)
    )

    if row['name'] == 'EMF422-1_string_2':
        if row['rotation'] == 0:
            strings_df.loc[idx, 'Type'] = "U_R"
        else:
            strings_df.loc[idx, 'Type'] = "U_L"

    elif mask_TMB.any():
        y_neighbors = strings_df.loc[mask_TMB, 'Y']

        if (y > y_neighbors).all():
            strings_df.loc[idx, 'Type'] = "T"
        elif (y > y_neighbors).any():
            strings_df.loc[idx, 'Type'] = "M"
        else:
            strings_df.loc[idx, 'Type'] = "B"

    else:
        strings_df.loc[idx, 'Type'] = "S"


strings_df.loc[:,'Line'] = np.where(
    (strings_df["Type"] == "S") | (strings_df["Type"] == "U_R") | (strings_df["Type"] == "U_L") | (strings_df["Type"] == "M"),
    round(strings_df["Y"]),
    np.where(
        strings_df["Type"] == "T",
        round(strings_df["Y"] - 2.3204),
        round(strings_df["Y"] + 2.3204)
    )
)


strings_df.sort_values(by=["Line", "X"], ascending=[False,True], inplace=True, ignore_index=True)
print(strings_df)
print(strings_df['Type'].value_counts())

               layer               name          X            Y UFV PST INV  \
0     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
1     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
2     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756677.6250  01  01  01   
3     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756675.2500  01  01  01   
4     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756672.9375  01  01  01   
...              ...                ...        ...          ...  ..  ..  ..   
5483  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5484  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5485  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5486  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5487  EMF_em_pst_3.3  EMF422-1_string_1  700151.75  755207.6250  03  03  20   

           GRP STR  rotation Type      Line  
0    

Conditions to break the strings into groups and subgroups.
=

In [7]:
#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Break in the Structure
condition2 = abs(strings_df["X"] - strings_df["X"].shift()) > 40

#Different PSTs
condition3 = strings_df["GRP"] != strings_df["GRP"].shift()



break_condition_1 = condition1 | condition2 | condition3
break_condition_2 = condition1 | condition2

strings_df['SubGroup'] = break_condition_1.cumsum()
strings_df['Group'] = break_condition_2.cumsum()
print(strings_df)

               layer               name          X            Y UFV PST INV  \
0     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
1     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
2     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756677.6250  01  01  01   
3     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756675.2500  01  01  01   
4     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756672.9375  01  01  01   
...              ...                ...        ...          ...  ..  ..  ..   
5483  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5484  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5485  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5486  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5487  EMF_em_pst_3.3  EMF422-1_string_1  700151.75  755207.6250  03  03  20   

           GRP STR  rotation Type      Line  SubGro

W exit means the west-most exit of the structure line.
E exit means the east-most exit of the structure line.

Also creates the Dataframe for the cable generation.

In [8]:
strings_df['W exit'] = np.where(
    strings_df["Type"] == "C",
    strings_df['X'] - 5.193,
    np.where(
        (strings_df["Type"] == "U_R") | (strings_df["Type"] == "U_L"),
        strings_df['X'] - 11.053,
        strings_df['X'] - 16.913
    )
)

strings_df['E exit'] = np.where(
    strings_df["Type"] == "C",
    strings_df['X'] + 5.193,
    np.where(
        (strings_df["Type"] == "U_R") | (strings_df["Type"] == "U_L"),
        strings_df['X'] + 11.053,
        strings_df['X'] + 16.913
    )
)

strings_df['SubGroup - Position X - W'] = strings_df.groupby("SubGroup")["W exit"].transform('min')
strings_df['SubGroup - Position X - E'] = strings_df.groupby("SubGroup")["E exit"].transform('max')

strings_df['Group - Position X - W'] = strings_df.groupby("Group")["W exit"].transform('min')
strings_df['Group - Position X - E'] = strings_df.groupby("Group")["E exit"].transform('max')

cables_df = strings_df.merge(inverter_df, on="GRP")

strings_df['No. of Strings'] = strings_df.groupby("SubGroup")['name'].transform('count')

print(strings_df)
print(strings_df['No. of Strings'].value_counts())
print(strings_df['No. of Strings'].value_counts().sum())

               layer               name          X            Y UFV PST INV  \
0     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
1     EMF_em_pst_1.1  EMF422-1_string_2  700354.75  756686.2500  01  01  01   
2     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756677.6250  01  01  01   
3     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756675.2500  01  01  01   
4     EMF_em_pst_1.1  EMF422-1_string_3  700361.25  756672.9375  01  01  01   
...              ...                ...        ...          ...  ..  ..  ..   
5483  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5484  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755228.6250  03  03  20   
5485  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5486  EMF_em_pst_3.3  EMF422-1_string_2  700151.75  755218.1250  03  03  20   
5487  EMF_em_pst_3.3  EMF422-1_string_1  700151.75  755207.6250  03  03  20   

           GRP STR  rotation  ...      Line  SubGro

Grouping the data per Subgroup

In [9]:
groups_df = strings_df.groupby("SubGroup")[['GRP','Line', 'SubGroup - Position X - W', 'SubGroup - Position X - E', 'Group - Position X - W', 'Group - Position X - E','W exit', 'E exit', 'No. of Strings']].first()
groups_df.rename(columns={'Line': 'Y', 'Group - Position X - W': 'X-W-STRUCT', 'Group - Position X - E': 'X-E-STRUCT', 'SubGroup - Position X - W': 'X-W-STR', 'SubGroup - Position X - E': 'X-E-STR'}, inplace=True)
print(f'Maximum number of strings in a group: {groups_df['No. of Strings'].max()}\n')
print(groups_df)

Maximum number of strings in a group: 22

               GRP         Y      X-W-STR      X-E-STR   X-W-STRUCT  \
SubGroup                                                              
1         01.01.01  756686.0  700343.6875  700365.8125  700343.6875   
2         01.01.01  756675.0  700344.3125  700401.6250  700344.3125   
3         01.01.01  756664.0  700344.9375  700437.4375  700344.9375   
4         01.01.01  756653.0  700345.6250  700473.2500  700345.6250   
5         01.01.01  756642.0  700346.2500  700380.1250  700346.2500   
...            ...       ...          ...          ...          ...   
626       03.03.20  755250.0  700069.1875  700173.3750  700069.1875   
627       03.03.20  755239.0  700111.1875  700168.5000  700111.1875   
628       03.03.20  755229.0  700140.6875  700162.8125  700140.6875   
629       03.03.20  755218.0  700140.6875  700162.8125  700140.6875   
630       03.03.20  755208.0  700134.8125  700168.6875  700134.8125   

           X-E-STRUCT       W exit

Duct_map is needed to represent how many ducts are necessary and how many strings are there in each duct. Given that the maximum of strings in a duct is 3, the duct_map is as follows:
=

In [10]:
duct_map = {
    1: '-1-',
    2: '-2-',
    3: '-3-',
    4: '-3-1-',
    5: '-3-2-',
    6: '-3-3-',
    7: '-3-3-1-',
    8: '-3-3-2-',
    9: '-3-3-3-',
    10: '-3-3-3-1-',
    11: '-3-3-3-2-',
    12: '-3-3-3-3-',
    13: '-3-3-3-3-1-',
    14: '-3-3-3-3-2-',
    15: '-3-3-3-3-3-',
    16: '-3-3-3-3-3-1-',
    17: '-3-3-3-3-3-2-',
    18: '-3-3-3-3-3-3-',
    19: '-3-3-3-3-3-3-1-',
    20: '-3-3-3-3-3-3-2-',
    21: '-3-3-3-3-3-3-3-',
    22: '-3-3-3-3-3-3-3-1-',
}

groups_df.loc[:,'Duct Definition'] = groups_df['No. of Strings'].map(duct_map)
groups_df['Number of Ducts'] = groups_df['Duct Definition'].astype(str).str.count('-')-1
groups_df = groups_df.merge(inverter_df, on='GRP')
groups_df['W or E'] = np.where(
    abs(groups_df['X-W-STRUCT'] - groups_df['inverter_x']) > abs(groups_df['X-E-STRUCT'] - groups_df['inverter_x']),
    "E",
    "W"
)

groups_df['closest_x'] = np.where(
    groups_df['W or E'] == "W",
    groups_df['X-W-STRUCT'],
    groups_df['X-E-STRUCT']
)
print(groups_df)

          GRP         Y      X-W-STR      X-E-STR   X-W-STRUCT   X-E-STRUCT  \
0    01.01.01  756686.0  700343.6875  700365.8125  700343.6875  700365.8125   
1    01.01.01  756675.0  700344.3125  700401.6250  700344.3125  700401.6250   
2    01.01.01  756664.0  700344.9375  700437.4375  700344.9375  700437.4375   
3    01.01.01  756653.0  700345.6250  700473.2500  700345.6250  700473.2500   
4    01.01.01  756642.0  700346.2500  700380.1250  700346.2500  700509.0625   
..        ...       ...          ...          ...          ...          ...   
623  03.03.20  755250.0  700069.1875  700173.3750  700069.1875  700173.3750   
624  03.03.20  755239.0  700111.1875  700168.5000  700111.1875  700168.5000   
625  03.03.20  755229.0  700140.6875  700162.8125  700140.6875  700162.8125   
626  03.03.20  755218.0  700140.6875  700162.8125  700140.6875  700162.8125   
627  03.03.20  755208.0  700134.8125  700168.6875  700134.8125  700168.6875   

          W exit       E exit  No. of Strings Duct 

In [11]:
cables_df.drop(columns=['UFV_x', 'UFV_y', 'PST_x', 'PST_y', 'INV_x', 'INV_y', 'SubGroup', 'Group', 'layer_x', 'name_x', 'layer_y', 'name_y'], inplace=True)
cables_df.rename(columns={'Group - Position X - W': 'X-W-STRUCT', 'Group - Position X - E': 'X-E-STRUCT', 'SubGroup - Position X - W': 'X-W-STR', 'SubGroup - Position X - E': 'X-E-STR'}, inplace=True)
cables_df['W or E or M'] = np.where(
    abs(cables_df['W exit'] - cables_df['inverter_x']) > abs(cables_df['E exit'] - cables_df['inverter_x']),
    "E",
    np.where(
        (cables_df['inverter_x'] < cables_df['W exit']) & (cables_df['inverter_x'] > cables_df['E exit']),
        "M",
        "W"
    )
)

# THIS FINDS THE CLOSEST X TO EXIT THE STRING.
# CALCULATING WHICH ONE IS THE CLOSEST ONE (X-W-STRUCT, X-E-STRUCT).
cables_df['closest_x'] = np.where(
    abs(cables_df['X-W-STRUCT'] - cables_df['inverter_x']) < abs(cables_df['X-E-STRUCT'] - cables_df['inverter_x']),
    cables_df['X-W-STRUCT'],
    cables_df['X-E-STRUCT']
)

cables_df['Exit'] = np.where(
    cables_df['W or E or M'] == 'E',
    cables_df['E exit'],
    cables_df['W exit']
)

cables_df.sort_values(['GRP', 'Line'], inplace = True)
print(cables_df.head())

              X            Y       GRP STR  rotation Type      Line  \
26  700363.1875  756644.6250  01.01.01  27       0.0    T  756642.0   
27  700363.1875  756642.2500  01.01.01  28       0.0    M  756642.0   
15  700362.5625  756655.6250  01.01.01  16       0.0    T  756653.0   
16  700362.5625  756653.2500  01.01.01  17       0.0    M  756653.0   
17  700362.5625  756650.9375  01.01.01  18       0.0    B  756653.0   

        W exit      E exit     X-W-STR     X-E-STR  X-W-STRUCT   X-E-STRUCT  \
26  700346.250  700380.125  700346.250  700380.125  700346.250  700509.0625   
27  700346.250  700380.125  700346.250  700380.125  700346.250  700509.0625   
15  700345.625  700379.500  700345.625  700473.250  700345.625  700473.2500   
16  700345.625  700379.500  700345.625  700473.250  700345.625  700473.2500   
17  700345.625  700379.500  700345.625  700473.250  700345.625  700473.2500   

    inverter_x  inverter_y W or E or M    closest_x        Exit  
26   700436.25   756667.25      

In [12]:
# print(cables_df.columns)
# print(groups_df.columns)
def position_of_structure_regarding_inverter(df):
    """
    The function takes a Dataframe and defines where the line of structure or the string is regarding their inverter.
    Option 1: Right (R)(X coordinate) and above (A)(Y coordinate) the inverter. Inverter east of the GRP (E). Being set as "R_A_E"; or
    Option 2: Right (R)(X coordinate) and below (B)(Y coordinate) the inverter. Inverter east of the GRP (E). Being set as "R_B_E"; or
    Option 3: Left (L)(X coordinate) and above (A)(Y coordinate) the inverter. Inverter east of the GRP (E). Being set as "L_A_E"; or
    Option 4: Left (L)(X coordinate) and below (B)(Y coordinate) the inverter. Inverter east of the GRP (E). Being set as "L_B_E"; or
    Option 5: Right (R)(X coordinate) and above (A)(Y coordinate) the inverter. Inverter west of the GRP (W). Being set as "R_A_W"; or
    Option 6: Right (R)(X coordinate) and below (B)(Y coordinate) the inverter. Inverter west of the GRP (W). Being set as "R_B_W"; or
    Option 7: Left (L)(X coordinate) and above (A)(Y coordinate) the inverter. Inverter west of the GRP (W). Being set as "L_A_W"; or
    Option 8: Left (L)(X coordinate) and below (B)(Y coordinate) the inverter. Inverter west of the GRP (W). Being set as "L_B_W".
    """
    df = df.merge(inverter_position, on='GRP')

    inverter_east = df['INVERTER - E or W'] == 'E'
    inverter_west = ~inverter_east

    right_of_inverter = df['closest_x'] - df['inverter_x'] > 0
    left_of_inverter = ~right_of_inverter

    above_inverter = df['Y'] - df['inverter_y'] > 0
    below_inverter = ~above_inverter

    conditions = [
        # East cases
        inverter_east & right_of_inverter & above_inverter,
        inverter_east & right_of_inverter & below_inverter,
        inverter_east & left_of_inverter & above_inverter,
        inverter_east & left_of_inverter & below_inverter,

        # West cases
        inverter_west & right_of_inverter & above_inverter,
        inverter_west & right_of_inverter & below_inverter,
        inverter_west & left_of_inverter & above_inverter,
        inverter_west & left_of_inverter & below_inverter,
    ]

    choices = ['R_A_E', 'R_B_E', 'L_A_E', 'L_B_E', 'R_A_W', 'R_B_W', 'L_A_W', 'L_B_W']


    df['Relative_Position'] = np.select(conditions, choices, default='unk')

    return df

cables_df = position_of_structure_regarding_inverter(cables_df)
groups_df = position_of_structure_regarding_inverter(groups_df)
cables_df.to_excel(r"C:\Users\Usuario\Desktop\cables.xlsx")
groups_df.to_excel(r"C:\Users\Usuario\Desktop\groups.xlsx")

In [13]:
script_df = groups_df[['No. of Strings', 'Duct Definition', 'Number of Ducts', 'closest_x', 'Y', 'inverter_x', 'inverter_y', 'Relative_Position']].copy()

def intermediary_coordinates(df):
    df['intermediary_y_1'] = np.where(
        df['inverter_y'] > df['Y'],
        df['inverter_y'] - 1.7,
        df['inverter_y'] + 1.7
    )

    df['intermediary_y_2'] = np.where(
        df['inverter_y'] > df['Y'],
        df['inverter_y'] - 1.5,
        df['inverter_y'] + 1.5
    )

    df['intermediary_y_3'] = np.where(
        df['inverter_y'] > df['Y'],
        df['inverter_y'] - 1.3,
        df['inverter_y'] + 1.3
    )

    df['inverter_y'] = np.where(
        df['inverter_y'] > df['Y'],
        df['inverter_y'] - 0.1,
        df['inverter_y'] + 0.1
    )

intermediary_coordinates(cables_df)
intermediary_coordinates(script_df)

script_df.rename(columns={'Y': 'Line'}, inplace=True)

'''
RED AND BLACK CABLES
This script generates the script according to the position of the inverter in relation to the structure, it has three patterns:

R_A_W, L_A_E = pattern 1
R_B_W, L_B_E = pattern 2
L_A_W, L_B_W, R_A_E, R_B_E = pattern 3

Pattern 1 generates a polyline that has 8 points:
1. End of the string
2. Pile coordinates
3. Pile X coordinates and Line of the Structure directly above the inverter
4. Inverter X and Line of the Structure directly above the inverter
5 to 7. Intermediary coordinates for stretching purposes
8. Inverter coordinates

Pattern 2 generates a polyline that has 8 points:
1. End of the string
2. Pile coordinates
3. Pile X coordinates and Line of the Structure directly below the inverter
4. Inverter X and Line of the Structure directly below the inverter
5 to 7. Intermediary coordinates for stretching purposes
8. Inverter coordinates

Pattern 3 generates a polyline that has 7 points:
1. End of the string
2. Pile coordinates
3. Inverter X and String Y coordinates
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates
'''

def make_pline(row, is_cable, offset=None):
    segments = []
    if is_cable:
        segments.extend([f"{row['Exit']},{row['Line']}"])

    segments.extend([f"{row['closest_x']},{row['Line']}"])

    if offset is not None:
        segments.extend([
            f"{row['closest_x']},{row['inverter_y'] + offset}",
            f"{row['inverter_x']},{row['inverter_y'] + offset}"
        ])
    else:
        segments.extend([f"{row['inverter_x']},{row['Line']}"])

    segments.extend([
        f"{row['inverter_x']},{row['intermediary_y_1']}",
        f"{row['inverter_x']},{row['intermediary_y_2']}",
        f"{row['inverter_x']},{row['intermediary_y_3']}",
        f"{row['inverter_x']},{row['inverter_y']} "
    ])

    return "_.pline " + " ".join(segments)

distance1 = 6.6553 # Distance between the inverter and the structure line above it
distance2 = 2 # Distance between the inverter and the structure line below it

cable_choice_R_A_W = cables_df.apply(lambda row: make_pline(row, True, distance1), axis=1)
cable_choice_R_B_W = cables_df.apply(lambda row: make_pline(row, True, -distance2), axis=1)
cable_choice_L_A_W = cables_df.apply(lambda row: make_pline(row, True), axis=1)
cable_choice_L_B_W = cables_df.apply(lambda row: make_pline(row, True), axis=1)
cable_choice_R_A_E = cables_df.apply(lambda row: make_pline(row, True), axis=1)
cable_choice_R_B_E = cables_df.apply(lambda row: make_pline(row, True), axis=1)
cable_choice_L_A_E = cables_df.apply(lambda row: make_pline(row, True, distance1), axis=1)
cable_choice_L_B_E = cables_df.apply(lambda row: make_pline(row, True, -distance2), axis=1)

trench_or_duct_choice_R_A_W = script_df.apply(lambda row: make_pline(row, False, distance1), axis=1)
trench_or_duct_choice_R_B_W = script_df.apply(lambda row: make_pline(row, False, -distance2), axis=1)
trench_or_duct_choice_L_A_W = script_df.apply(lambda row: make_pline(row, False), axis=1)
trench_or_duct_choice_L_B_W = script_df.apply(lambda row: make_pline(row, False), axis=1)
trench_or_duct_choice_R_A_E = script_df.apply(lambda row: make_pline(row, False), axis=1)
trench_or_duct_choice_R_B_E = script_df.apply(lambda row: make_pline(row, False), axis=1)
trench_or_duct_choice_L_A_E = script_df.apply(lambda row: make_pline(row, False, distance1), axis=1)
trench_or_duct_choice_L_B_E = script_df.apply(lambda row: make_pline(row, False, distance2), axis=1)


cable_conditions = [cables_df['Relative_Position'] == 'R_A_W',
             cables_df['Relative_Position'] == 'R_B_W',
             cables_df['Relative_Position'] == 'L_A_W',
             cables_df['Relative_Position'] == 'L_B_W',
             cables_df['Relative_Position'] == 'R_A_E',
             cables_df['Relative_Position'] == 'R_B_E',
             cables_df['Relative_Position'] == 'L_A_E',
             cables_df['Relative_Position'] == 'L_B_E']

cable_choices = [cable_choice_R_A_W,
                 cable_choice_R_B_W,
                 cable_choice_L_A_W,
                 cable_choice_L_B_W,
                 cable_choice_R_A_E,
                 cable_choice_R_B_E,
                 cable_choice_L_A_E,
                 cable_choice_L_B_E]

cables_df['Script Cabos'] = np.select(cable_conditions, cable_choices, default='unk')
cables_df.to_excel(r"C:\Users\Usuario\Desktop\cables.xlsx")

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Cabos - Reask.scr", 'w') as f:
    f.write('PLINEWID 0\n')
    f.write("LAYER S EMF_cabo_solar_positivo\n\n")
    f.write('\n'.join(cables_df["Script Cabos"].astype(str)) + '\n')
    f.write("LAYER S EMF_cabo_solar_negativo\n\n")
    f.write('\n'.join(cables_df["Script Cabos"].astype(str)) + '\n')


'''
TRENCHES
This script generates the script according to the position of the inverter in relation to the structure, it has three patterns:

R_A_W, L_A_E = pattern 1
R_B_W, L_B_E = pattern 2
L_A_W, L_B_W, R_A_E, R_B_E = pattern 3

Pattern 1 generates a polyline that has 7 points:
1. Pile coordinates
2. Pile X coordinates and Line of the Structure directly above the inverter
3. Inverter X and Line of the Structure directly above the inverter
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates

Pattern 2 generates a polyline that has 7 points:
1. Pile coordinates
2. Pile X coordinates and Line of the Structure directly below the inverter
3. Inverter X and Line of the Structure directly below the inverter
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates

Pattern 3 generates a polyline that has 6 points:
1. Pile coordinates
2. Inverter X and String Y coordinates
3 to 5. Intermediary coordinates for stretching purposes
6. Inverter coordinates
'''

trench_or_duct_conditions = [script_df['Relative_Position'] == 'R_A_W',
             script_df['Relative_Position'] == 'R_B_W',
             script_df['Relative_Position'] == 'L_A_W',
             script_df['Relative_Position'] == 'L_B_W',
             script_df['Relative_Position'] == 'R_A_E',
             script_df['Relative_Position'] == 'R_B_E',
             script_df['Relative_Position'] == 'L_A_E',
             script_df['Relative_Position'] == 'L_B_E']

trench_choices = [trench_or_duct_choice_R_A_W,
                  trench_or_duct_choice_R_B_W,
                  trench_or_duct_choice_L_A_W,
                  trench_or_duct_choice_L_B_W,
                  trench_or_duct_choice_R_A_E,
                  trench_or_duct_choice_R_B_E,
                  trench_or_duct_choice_L_A_E,
                  trench_or_duct_choice_L_B_E]

script_df['Script Valas'] = np.select(trench_or_duct_conditions, trench_choices, default='unk')
script_df.to_excel(r"C:\Users\Usuario\Desktop\script.xlsx")


with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Valas - Reask.scr", 'w') as f:
    f.write('PLINEWID 0.5\n')
    f.write("LAYER S EMF_DC_vala\n\n")
    f.write('\n'.join(script_df["Script Valas"].astype(str)) + '\n')


'''
DUCTS
This script generates the script according to the position of the inverter in relation to the structure, it has three patterns:

R_A_W, L_A_E = pattern 1
R_B_W, L_B_E = pattern 2
L_A_W, L_B_W, R_A_E, R_B_E = pattern 3

Pattern 1 generates a polyline that has 7 points:
1. Pile coordinates
2. Pile X coordinates and Line of the Structure directly above the inverter
3. Inverter X and Line of the Structure directly above the inverter
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates

Pattern 2 generates a polyline that has 7 points:
1. Pile coordinates
2. Pile X coordinates and Line of the Structure directly below the inverter
3. Inverter X and Line of the Structure directly below the inverter
4 to 6. Intermediary coordinates for stretching purposes
7. Inverter coordinates

Pattern 3 generates a polyline that has 6 points:
1. Pile coordinates
2. Inverter X and String Y coordinates
3 to 5. Intermediary coordinates for stretching purposes
6. Inverter coordinates

For Script Dutos 3, it also has the multiplication to account for the number of ducts with 3 strings that have the same coordinates.
'''

duct_choices = [trench_or_duct_choice_R_A_W,
                  trench_or_duct_choice_R_B_W,
                  trench_or_duct_choice_L_A_W,
                  trench_or_duct_choice_L_B_W,
                  trench_or_duct_choice_R_A_E,
                  trench_or_duct_choice_R_B_E,
                  trench_or_duct_choice_L_A_E,
                  trench_or_duct_choice_L_B_E]


script_df['Script Dutos 1'] = np.select(trench_or_duct_conditions, duct_choices, default='unk')
script_df['Script Dutos 2'] = np.select(trench_or_duct_conditions, duct_choices, default='unk')
script_df['Script Dutos 3'] = np.select(trench_or_duct_conditions, duct_choices, default='unk')

script_df['Script Dutos 1'] = script_df.apply(
    lambda row: (row['Script Dutos 1']) * str(row['Duct Definition']).count('1'),
    axis=1
)
script_df['Script Dutos 2'] = script_df.apply(
    lambda row: (row['Script Dutos 2']) * str(row['Duct Definition']).count('2'),
    axis=1
)

script_df['Script Dutos 3'] = script_df.apply(
    lambda row: (row['Script Dutos 3'] + "\n") * str(row['Duct Definition']).count('3'),
    axis=1
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Dutos DC\Script Dutos - Reask.scr", 'w') as f:
    f.write('PLINEWID 0.03\n')
    # Layer 1
    layer1_commands = script_df["Script Dutos 1"].astype(str)
    layer1_commands = layer1_commands[layer1_commands != '']  # Remove empty strings
    if len(layer1_commands) > 0:  # Only write layer if there are commands
        f.write("LAYER S EMF_bt_eletroduto_1S\n\n")
        f.write('\n'.join(layer1_commands) + '\n')

    # Layer 2
    layer2_commands = script_df["Script Dutos 2"].astype(str)
    layer2_commands = layer2_commands[layer2_commands != '']
    if len(layer2_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_2S\n\n")
        f.write('\n'.join(layer2_commands) + '\n')

    # Layer 3
    layer3_commands = script_df["Script Dutos 3"].astype(str)
    layer3_commands = layer3_commands[layer3_commands != '']
    if len(layer3_commands) > 0:
        f.write("LAYER S EMF_bt_eletroduto_3S\n\n")
        f.write(''.join(layer3_commands) + '\n')


conn.close()